# **Load the Dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install --upgrade objaverse
!pip install torch torchvision torchaudio trimesh numpy objaverse tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.0 MB/s eta 0:00:00
  Created wheel for gputil: filename=GPUtil-1.4.0-py3-none-any.whl size=7392 sha256=4e2f6ba93d1a15b3f765349202a2fb89849cfefc8730d0f6c3aa58c1b84b6787
  Stored in directory: /root/.cache/pip/wheels/2b/4d/8f/55fb4f7b9b591891e8d3f72977c4ec6c7763b39c19f0861595
Successfully built gputil
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import numpy as np
import objaverse
import os
import trimesh
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import logging
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import CosineAnnealingLR
import gc

In [ ]:
uuid_embeddings = np.load("/content/drive/MyDrive/1_Diamond/10623/df_english.npy", allow_pickle=True)

In [ ]:
uuid_embeddings

array([['bfd459469a9449978ff5fc7199d5848b',
        "3D Mandala #2 - The Power of Love went a bit weird with this one.. in the centre a blue female is punching the red male, breaking his heart in the process, triggering feelings like he's being anally probed by aliens, sending shocks through his eyeball, releasing some tears."],
       ['a4bb737d810049c98680a53262552a7b',
        "Glass 眼睛 A Glass which don't have shader and i don't know why"],
       ['01474f78a328473b8d99ded65effb2cc',
        'Low Poly Barrel super simple barrel'],
       ...,
       ['69925e5dab364b4ba87b036f6042c442',
        'Consolevert4 SciFi Controls \n\nFree to use in commercial and personal projects!\n\nEnjoy!'],
       ['63b0f1e3ccdd4b3fabec4443a2b25cd8',
        'Neo-Normcore Collection, Insulin Placebo Human Insulin Hexamer by model3dbiology is licensed under CC Attribution'],
       ['168bb415db3247a19e5fcc4d23c26c91',
        "R.O.Y. Bot Zbrush, Go-Z'd to C4D create OVDB Bevels and baked in 3D Coat. Sci

In [ ]:
uuid_embeddings.shape

(236277, 2)

In [ ]:
uuid_embeddings[0, 1]

"3D Mandala #2 - The Power of Love went a bit weird with this one.. in the centre a blue female is punching the red male, breaking his heart in the process, triggering feelings like he's being anally probed by aliens, sending shocks through his eyeball, releasing some tears."

# **Load the Components**

In [ ]:
import numpy as np
import objaverse
import os
import trimesh
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm import tqdm
import logging
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import CosineAnnealingLR
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def generate_triplane_from_mesh(obj_path, triplane_resolution=128, voxel_resolution=128, device='cuda'):
    try:
        mesh = trimesh.load(obj_path, force='mesh', process=False)
        if isinstance(mesh, trimesh.Scene):
            mesh = mesh.dump(concatenate=True)
        if not isinstance(mesh, trimesh.Trimesh):
            logger.error(f"Invalid mesh at {obj_path}: not a Trimesh")
            return None
        if len(mesh.vertices) == 0 or len(mesh.faces) == 0:
            logger.error(f"Empty mesh at {obj_path}")
            return None

        center = mesh.bounds.mean(axis=0)
        mesh.apply_translation(-center)
        max_extent = np.ptp(mesh.bounds, axis=0).max()
        if max_extent > 1e-6:
            mesh.apply_scale(1.0 / max_extent)

        pitch = 1.0 / voxel_resolution
        voxel_grid = mesh.voxelized(pitch=pitch)
        voxel_matrix = voxel_grid.matrix.astype(np.float32)

        current_shape = voxel_matrix.shape
        target_shape = (voxel_resolution, voxel_resolution, voxel_resolution)
        padded_matrix = np.zeros(target_shape, dtype=np.float32)
        min_shape = tuple(min(s, t) for s, t in zip(current_shape, target_shape))
        padded_matrix[:min_shape[0], :min_shape[1], :min_shape[2]] = \
            voxel_matrix[:min_shape[0], :min_shape[1], :min_shape[2]]
        voxel_matrix = padded_matrix

        if voxel_matrix.sum() == 0:
            logger.error(f"Empty voxel grid for {obj_path}")
            return None

        plane_xy = np.max(voxel_matrix, axis=2)
        plane_yz = np.max(voxel_matrix, axis=0)
        plane_xz = np.max(voxel_matrix, axis=1)

        plane_xy_t = torch.from_numpy(plane_xy).unsqueeze(0).float().to(device)
        plane_yz_t = torch.from_numpy(plane_yz).unsqueeze(0).float().to(device)
        plane_xz_t = torch.from_numpy(plane_xz).unsqueeze(0).float().to(device)

        target_size = (triplane_resolution, triplane_resolution)
        plane_xy_t = F.interpolate(plane_xy_t.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0)
        plane_yz_t = F.interpolate(plane_yz_t.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0)
        plane_xz_t = F.interpolate(plane_xz_t.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0)

        plane_xy_t = torch.clamp(plane_xy_t, 0.0, 1.0)
        plane_yz_t = torch.clamp(plane_yz_t, 0.0, 1.0)
        plane_xz_t = torch.clamp(plane_xz_t, 0.0, 1.0)

        return plane_xy_t.cpu(), plane_yz_t.cpu(), plane_xz_t.cpu()
    except Exception as e:
        logger.error(f"Failed to process mesh at {obj_path}: {e}")
        return None

def generate_and_save_triplanes(caption_data_path, output_dir=None, start_index=0, end_index=None, triplane_resolution=128, voxel_resolution=128, device='cuda'):
    if output_dir is None:
        output_dir = os.path.dirname(caption_data_path)
    os.makedirs(output_dir, exist_ok=True)

    try:
        caption_data = np.load(caption_data_path, allow_pickle=True)
        logger.info(f"Loaded caption data with shape: {caption_data.shape}")
    except Exception as e:
        logger.error(f"Failed to load caption data: {e}")
        return

    max_index = caption_data.shape[0]
    if start_index < 0 or start_index >= max_index:
        logger.error(f"start_index {start_index} is out of bounds [0, {max_index})")
        return
    if end_index is None:
        end_index = max_index
    if end_index <= start_index or end_index > max_index:
        logger.error(f"end_index {end_index} is invalid; must be > {start_index} and <= {max_index}")
        return

    uuids = caption_data[start_index:end_index, 0]
    captions = caption_data[start_index:end_index, 1]

    for uuid, caption in tqdm(zip(uuids, captions), total=len(uuids), desc=f"Generating triplanes [{start_index}:{end_index}]"):
        try:
            objects = objaverse.load_objects(uids=[uuid])
            if uuid not in objects:
                logger.warning(f"Model {uuid} not found in Objaverse")
                continue
            obj_path = objects[uuid]
            if not isinstance(obj_path, str) or not os.path.exists(obj_path):
                logger.error(f"Invalid or missing file for {uuid}: {obj_path}")
                continue

            triplane = generate_triplane_from_mesh(obj_path, triplane_resolution, voxel_resolution, device)
            if triplane is None:
                logger.warning(f"Failed to generate triplane for {uuid}")
                continue

            plane_xy, plane_yz, plane_xz = triplane
            data = {
                'plane_xy': plane_xy.numpy(),
                'plane_yz': plane_yz.numpy(),
                'plane_xz': plane_xz.numpy(),
                'caption': caption
            }

            output_path = os.path.join(output_dir, f"{uuid}.npy")
            np.save(output_path, data)
            logger.info(f"Saved triplane and caption for {uuid} to {output_path}")

            del triplane, plane_xy, plane_yz, plane_xz, data
            torch.cuda.empty_cache() if device == 'cuda' else None

        except Exception as e:
            logger.error(f"Failed to process model {uuid}: {e}")
            continue

class TriplaneTextFromFilesDataset(Dataset):
    def __init__(self, triplane_dir, num_models=None, device='cuda'):
        self.triplane_dir = triplane_dir
        self.device = device

        self.npy_files = [f for f in os.listdir(triplane_dir) if f.endswith('.npy')]
        if not self.npy_files:
            logger.error(f"No .npy files found in {triplane_dir}")
            raise ValueError(f"No .npy files found in {triplane_dir}")

        if num_models is not None:
            self.npy_files = self.npy_files[:min(num_models, len(self.npy_files))]

        self.valid_files = []
        self.uuids = []

        for npy_file in tqdm(self.npy_files, desc="Validating .npy files"):
            try:
                file_path = os.path.join(self.triplane_dir, npy_file)
                data = np.load(file_path, allow_pickle=True).item()
                if not all(key in data for key in ['plane_xy', 'plane_yz', 'plane_xz', 'caption']):
                    logger.warning(f"Invalid .npy file {npy_file}: missing required keys")
                    continue
                if data['plane_xy'].shape != (1, 128, 128) or \
                   data['plane_yz'].shape != (1, 128, 128) or \
                   data['plane_xz'].shape != (1, 128, 128):
                    logger.warning(f"Invalid .npy file {npy_file}: incorrect triplane shapes")
                    continue
                self.valid_files.append(npy_file)
                self.uuids.append(npy_file.replace('.npy', ''))
            except Exception as e:
                logger.warning(f"Failed to load {npy_file}: {e}")
                continue

        if not self.valid_files:
            logger.error(f"No valid .npy files found in {triplane_dir}")
            raise ValueError(f"No valid .npy files found in {triplane_dir}")

        logger.info(f"Initialized dataset with {len(self.valid_files)} valid samples")

    def __len__(self):
        return len(self.valid_files)

    def __getitem__(self, idx):
        npy_file = self.valid_files[idx]
        file_path = os.path.join(self.triplane_dir, npy_file)

        try:
            data = np.load(file_path, allow_pickle=True).item()
            plane_xy = torch.from_numpy(data['plane_xy']).float().to(self.device)
            plane_yz = torch.from_numpy(data['plane_yz']).float().to(self.device)
            plane_xz = torch.from_numpy(data['plane_xz']).float().to(self.device)
            caption = str(data['caption'])
            return (plane_xy, plane_yz, plane_xz), caption
        except Exception as e:
            logger.error(f"Failed to load {npy_file}: {e}")
            raise e

class PositionalEncoding2D(nn.Module):
    def __init__(self, d_model, height, width):
        super().__init__()
        if d_model % 4 != 0:
            raise ValueError(f"Cannot use sin/cos positional encoding with odd dimension (got dim={d_model})")
        pe = torch.zeros(d_model, height, width)
        d_model_h = d_model // 2
        d_model_w = d_model // 2
        div_term = torch.exp(torch.arange(0., d_model_h, 2) * -(torch.log(torch.tensor(10000.0)) / d_model_h))
        pos_w = torch.arange(0., width).unsqueeze(1)
        pos_h = torch.arange(0., height).unsqueeze(1)
        pe[0:d_model_h:2, :, :] = torch.sin(pos_h * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)
        pe[1:d_model_h:2, :, :] = torch.cos(pos_h * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)
        pe[d_model_h::2, :, :] = torch.sin(pos_w * div_term).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
        pe[d_model_h+1::2, :, :] = torch.cos(pos_w * div_term).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :, :x.size(2), :x.size(3)]
        return x

class TriplaneDecoder(nn.Module):
    def __init__(self, encoder_dim=384, decoder_dim=512, decoder_layers=4, decoder_heads=8,
                 output_channels=3, output_resolution=128, input_patch_grid_res=16):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.decoder_dim = decoder_dim
        self.output_resolution = output_resolution
        self.input_patch_grid_res = input_patch_grid_res
        self.num_patches = input_patch_grid_res * input_patch_grid_res

        self.input_proj = nn.Linear(encoder_dim, decoder_dim * self.num_patches)
        self.pos_encoder = PositionalEncoding2D(decoder_dim, input_patch_grid_res, input_patch_grid_res)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=decoder_dim, nhead=decoder_heads, dim_feedforward=decoder_dim * 4,
            dropout=0.1, activation=F.gelu, batch_first=True, norm_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=decoder_layers)

        num_upsample_stages = int(torch.log2(torch.tensor(output_resolution // input_patch_grid_res)))
        if input_patch_grid_res * (2**num_upsample_stages) != output_resolution:
            raise ValueError("Output resolution must be a power-of-2 multiple of input_patch_grid_res")

        upsample_layers = []
        current_dim = decoder_dim
        for i in range(num_upsample_stages):
            out_dim = max(decoder_dim // (2**(i+1)), output_channels * 2)
            upsample_layers.append(nn.Sequential(
                nn.ConvTranspose2d(current_dim, out_dim, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_dim), nn.GELU(),
                nn.Conv2d(out_dim, out_dim, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_dim), nn.GELU()
            ))
            current_dim = out_dim
        self.upsample_neck = nn.Sequential(*upsample_layers)

        self.output_proj = nn.Conv2d(current_dim, output_channels, kernel_size=1, stride=1, padding=0)
        self.output_activation = nn.Sigmoid()

    def forward(self, text_embedding):
        batch_size = text_embedding.shape[0]
        seq = self.input_proj(text_embedding).view(batch_size, self.num_patches, self.decoder_dim)
        spatial_input = seq.permute(0, 2, 1).reshape(
            batch_size, self.decoder_dim, self.input_patch_grid_res, self.input_patch_grid_res)
        spatial_input_with_pos = self.pos_encoder(spatial_input)
        seq_with_pos = spatial_input_with_pos.flatten(2).permute(0, 2, 1)
        refined_seq = self.transformer_decoder(tgt=seq_with_pos, memory=seq_with_pos)
        spatial_features = refined_seq.permute(0, 2, 1).reshape(
            batch_size, self.decoder_dim, self.input_patch_grid_res, self.input_patch_grid_res)
        upsampled_features = self.upsample_neck(spatial_features)
        output_logits = self.output_proj(upsampled_features)
        output_triplane = self.output_activation(output_logits)

        plane_xy = output_triplane[:, 0:1, :, :]
        plane_yz = output_triplane[:, 1:2, :, :]
        plane_xz = output_triplane[:, 2:3, :, :]

        return plane_xy, plane_yz, plane_xz

class TextToTriplaneModel(nn.Module):
    def __init__(self, text_encoder_name='all-MiniLM-L6-v2', decoder_dim=512, decoder_layers=4,
                 decoder_heads=8, output_resolution=128):
        super().__init__()
        self.text_encoder = SentenceTransformer(text_encoder_name)
        self.text_encoder.eval()
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        self.decoder = TriplaneDecoder(
            encoder_dim=384,
            decoder_dim=decoder_dim,
            decoder_layers=decoder_layers,
            decoder_heads=decoder_heads,
            output_channels=3,
            output_resolution=output_resolution,
            input_patch_grid_res=16
        )

    def forward(self, captions, device):
        with torch.no_grad():
            text_embeddings = self.text_encoder.encode(captions, convert_to_tensor=True, device=device)
        plane_xy, plane_yz, plane_xz = self.decoder(text_embeddings)
        return plane_xy, plane_yz, plane_xz

def plot_sample(triplane, caption, uuid, title="Triplane"):
    try:
        if any(ord(char) > 127 for char in caption):
            try:
                plt.rcParams['font.family'] = 'Noto Sans CJK JP'
            except:
                logger.warning("Noto Sans CJK JP font not found, using default.")
                plt.rcParams['font.family'] = 'DejaVu Sans'
        else:
            plt.rcParams['font.family'] = 'DejaVu Sans'

        logger.info(f"Generating plot: {uuid}_{title.replace(' ', '_').lower()}.png")

        plane_xy, plane_yz, plane_xz = triplane
        xy_plane = plane_xy.cpu().numpy()[0]
        yz_plane = plane_yz.cpu().numpy()[0]
        xz_plane = plane_xz.cpu().numpy()[0]
        xy_plane = np.clip(xy_plane, 0, 1)
        yz_plane = np.clip(yz_plane, 0, 1)
        xz_plane = np.clip(xz_plane, 0, 1)
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(xy_plane, cmap='gray')
        axes[0].set_title(f"XY Plane\nUUID: {uuid}")
        axes[0].axis('off')
        axes[1].imshow(yz_plane, cmap='gray')
        axes[1].set_title("YZ Plane")
        axes[1].axis('off')
        axes[2].imshow(xz_plane, cmap='gray')
        axes[2].set_title("XZ Plane")
        axes[2].axis('off')
        fig.suptitle(f"{title}\nCaption: {caption}", fontsize=12, y=0.05)
        plt.tight_layout()
        plt.savefig(f"{uuid}_{title.replace(' ', '_').lower()}.png")
        plt.show()
        plt.close()

        plt.rcParams['font.family'] = 'DejaVu Sans'
    except Exception as e:
        logger.error(f"Error plotting sample: {e}")

def plot_triplane_grid(pred_triplane, gt_triplane, caption, uuid, title="Triplane Comparison"):
    try:
        if any(ord(char) > 127 for char in caption):
            try:
                plt.rcParams['font.family'] = 'Noto Sans CJK JP'
            except:
                logger.warning("Noto Sans CJK JP font not found, using default.")
                plt.rcParams['font.family'] = 'DejaVu Sans'
        else:
            plt.rcParams['font.family'] = 'DejaVu Sans'

        logger.info(f"Generating grid plot: {uuid}_{title.replace(' ', '_').lower()}.png with caption: {caption}")

        pred_xy, pred_yz, pred_xz = pred_triplane
        gt_xy, gt_yz, gt_xz = gt_triplane

        pred_xy = np.clip(pred_xy.cpu().numpy()[0, 0], 0, 1)
        pred_yz = np.clip(pred_yz.cpu().numpy()[0, 0], 0, 1)
        pred_xz = np.clip(pred_xz.cpu().numpy()[0, 0], 0, 1)
        gt_xy = np.clip(gt_xy.cpu().numpy()[0], 0, 1)
        gt_yz = np.clip(gt_yz.cpu().numpy()[0], 0, 1)
        gt_xz = np.clip(gt_xz.cpu().numpy()[0], 0, 1)

        fig, axes = plt.subplots(2, 3, figsize=(15, 10))

        axes[0, 0].imshow(pred_xy, cmap='gray')
        axes[0, 0].set_title("Predicted XY Plane")
        axes[0, 0].axis('off')
        axes[0, 1].imshow(pred_yz, cmap='gray')
        axes[0, 1].set_title("Predicted YZ Plane")
        axes[0, 1].axis('off')
        axes[0, 2].imshow(pred_xz, cmap='gray')
        axes[0, 2].set_title("Predicted XZ Plane")
        axes[0, 2].axis('off')

        axes[1, 0].imshow(gt_xy, cmap='gray')
        axes[1, 0].set_title("Ground Truth XY Plane")
        axes[1, 0].axis('off')
        axes[1, 1].imshow(gt_yz, cmap='gray')
        axes[1, 1].set_title("Ground Truth YZ Plane")
        axes[1, 1].axis('off')
        axes[1, 2].imshow(gt_xz, cmap='gray')
        axes[1, 2].set_title("Ground Truth XZ Plane")
        axes[1, 2].axis('off')

        fig.suptitle(f"{title}\nCaption: {caption}\nUUID: {uuid}", fontsize=12, y=0.02)
        plt.tight_layout()
        plt.savefig(f"{uuid}_{title.replace(' ', '_').lower()}.png")
        plt.show()
        plt.close()

        plt.rcParams['font.family'] = 'DejaVu Sans'
    except Exception as e:
        logger.error(f"Error plotting triplane grid: {e}")
        raise e

def train_model(model, train_loader, val_loader, npy_file_path, num_epochs=10, lr=1e-3, device='cuda', checkpoint_dir='.'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.L1Loss()

    best_val_loss = float('inf')
    best_model_path = os.path.join(checkpoint_dir, "best_text_to_triplane_model.pth")

    os.makedirs(checkpoint_dir, exist_ok=True)

    logger.info(f"Loading .npy file: {npy_file_path}")
    try:
        data = np.load(nnot all(key in data for key in ['plane_xy', 'plane_yz', 'plane_xz', 'caption']):
            raise ValueError(f"Invalid .npy file {npy_file_path}: missing required keys")
        if data['plane_xy'].shape != (1, 128, 128) or \
           data['plane_yz'].shape != (1, 128, 128) or \
           data['plane_xz'].shape != (1, 128, 128):
            raise ValueError(f"Invalid .npy file {npy_file_path}: incorrect triplane shapes")
        npy_caption = str(data['caption'])
        npy_gt_xy = torch.from_numpy(data['plane_xy']).float()
        npy_gt_yz = torch.from_numpy(data['plane_yz']).float()
        npy_gt_xz = torch.from_numpy(data['plane_xz']).float()
        logger.info(f"Successfully loaded .npy file with caption: {npy_caption}")
    except Exception as e:
        logger.error(f"Failed to load {npy_file_path}: {e}")
        raise e

    prev_pred_triplane = None

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for triplane, caption in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
            triplane_xy, triplane_yz, triplane_xz = triplane
            triplane_xy = triplane_xy.to(device)
            triplane_yz = triplane_yz.to(device)
            triplane_xz = triplane_xz.to(device)
            optimizer.zero_grad()
            output_xy, output_yz, output_xz = model(caption, device=device)
            loss_xy = criterion(output_xy, triplane_xy)
            loss_yz = criterion(output_yz, triplane_yz)
            loss_xz = criterion(output_xz, triplane_xz)
            loss = (loss_xy + loss_yz + loss_xz) / 3
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            torch.cuda.empty_cache() if device == 'cuda' else None
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for triplane, caption in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"):
                triplane_xy, triplane_yz, triplane_xz = triplane
                triplane_xy = triplane_xy.to(device)
                triplane_yz = triplane_yz.to(device)
                triplane_xz = triplane_xz.to(device)
                output_xy, output_yz, output_xz = model(caption, device=device)
                loss_xy = criterion(output_xy, triplane_xy)
                loss_yz = criterion(output_yz, triplane_yz)
                loss_xz = criterion(output_xz, triplane_xz)
                loss = (loss_xy + loss_yz + loss_xz) / 3
                val_loss += loss.item()
        val_loss /= len(val_loader)

        scheduler.step()

        logger.info(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            logger.info(f"Saved best model with Val Loss: {best_val_loss:.4f}")

        if (epoch + 1) % 10 == 0:
            checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pth")
            torch.save(model.state_dict(), checkpoint_path)
            logger.info(f"Saved checkpoint at epoch {epoch+1} to {checkpoint_path}")

            logger.info(f"Generating triplanes for .npy file caption at epoch {epoch+1}")
            model.eval()
            with torch.no_grad():
                output_xy, output_yz, output_xz = model([npy_caption], device=device)
                pred_triplane = (output_xy, output_yz, output_xz)
                gt_triplane = (npy_gt_xy, npy_gt_yz, npy_gt_xz)

                if prev_pred_triplane is not None:
                    diff_xy = torch.mean(torch.abs(output_xy - prev_pred_triplane[0])).item()
                    diff_yz = torch.mean(torch.abs(output_yz - prev_pred_triplane[1])).item()
                    diff_xz = torch.mean(torch.abs(output_xz - prev_pred_triplane[2])).item()
                    logger.info(f"Prediction L1 diff between epoch {epoch+1} and {epoch-9}: "
                                f"XY={diff_xy:.6f}, YZ={diff_yz:.6f}, XZ={diff_xz:.6f}")

                prev_pred_triplane = (output_xy.clone(), output_yz.clone(), output_xz.clone())

                plot_triplane_grid(
                    pred_triplane=pred_triplane,
                    gt_triplane=gt_triplane,
                    caption=npy_caption,
                    uuid=f"inference_{os.path.basename(npy_file_path).replace('.npy', '')}_epoch_{epoch+1}",
                    title=f"Triplane Comparison Epoch {epoch+1}"
                )

    return model, best_model_path

def load_model_checkpoint(checkpoint_path, device='cuda', train_mode=False):
    logger = logging.getLogger(__name__)
    try:
        model = TextToTriplaneModel(
            text_encoder_name='all-MiniLM-L6-v2',
            decoder_dim=512,
            decoder_layers=4,
            decoder_heads=8,
            output_resolution=128
        )
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint)
        logger.info(f"Successfully loaded checkpoint from {checkpoint_path}")
        model = model.to(device)
        if train_mode:
            model.train()
            logger.info("Model set to training mode")
        else:
            model.eval()
            logger.info("Model set to evaluation mode")
        return model
    except FileNotFoundError:
        logger.error(f"Checkpoint file not found: {checkpoint_path}")
        raise
    except Exception as e:
        logger.error(f"Failed to load checkpoint from {checkpoint_path}: {e}")
        raise

def infer_triplane(model, caption, device='cuda'):
    model.eval()
    uuid = f"inference_{hash(caption) % 1000000}"
    with torch.no_grad():
        output_xy, output_yz, output_xz = model([caption], device=device)
        triplane = (output_xy[0], output_yz[0], output_xz[0])
        plot_sample(triplane, caption, uuid, title="Inferred Triplane")
        logger.info(f"Plotted inferred triplane for caption: {caption}")
    return triplane

# **Configurations**

In [ ]:
# Configuration
CAPTION_DATA_PATH = "/content/drive/MyDrive/1_Diamond/10623/df_english.npy"
TRIPLANE_DIR = "/content/drive/MyDrive/1_Diamond/10623/triplanes"
CHECKPOINT_DIR = "/content/drive/MyDrive/1_Diamond/10623/checkpoints"  # New directory for checkpoints
NPY_FILE_PATH = "/content/drive/MyDrive/1_Diamond/10623/triplanes/fecabaeaf1794756b45333b97d6e2374.npy"
START_INDEX = 648
END_INDEX = 700
NUM_MODELS = 4000
TRAIN_SPLIT = 0.75
VAL_SPLIT = 0.20
TEST_SPLIT = 0.05
BATCH_SIZE = 1
NUM_EPOCHS = 100
LEARNING_RATE = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# **Create the dataset**

In [ ]:
generate_and_save_triplanes(
    caption_data_path=CAPTION_DATA_PATH,
    output_dir=TRIPLANE_DIR,
    start_index=START_INDEX,
    end_index=END_INDEX,
    triplane_resolution=128,
    voxel_resolution=128,
    device=DEVICE
)

def count_files(directory):
    return len([f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))])

num_files = count_files(TRIPLANE_DIR)
print(f"Number of files in {TRIPLANE_DIR}: {num_files}")

# **Load the Dataset**

In [ ]:
# Create dataset
dataset = TriplaneTextFromFilesDataset(
    triplane_dir=TRIPLANE_DIR,
    num_models=NUM_MODELS,
    device=DEVICE
)

# Split dataset
train_size = int(TRAIN_SPLIT * len(dataset))
val_size = int(VAL_SPLIT * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


Validating .npy files: 100%|██████████| 3000/3000 [00:28<00:00, 105.41it/s]


# **Load the model and train**

In [ ]:
best_model_path = f"{CHECKPOINT_DIR}/best_text_to_triplane_model.pth"
model = load_model_checkpoint(best_model_path, device=DEVICE, train_mode=False)

In [ ]:
model = TextToTriplaneModel(
    text_encoder_name='all-MiniLM-L6-v2',
    decoder_dim=512,
    decoder_layers=4,
    decoder_heads=8,
    output_resolution=128
)

trained_model, best_model_path = train_model(
    model, train_loader, val_loader,
    npy_file_path=NPY_FILE_PATH,
    num_epochs=NUM_EPOCHS, lr=LEARNING_RATE, device=DEVICE,
    checkpoint_dir=CHECKPOINT_DIR
)

# **Test the model**

In [ ]:
caption = "A Gun"
triplane = infer_triplane(model, caption, device=DEVICE)

# **Chamfer Distance calculation**

In [ ]:
!pip install trimesh scipy plotly transformers --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import numpy as np
import os
import math
import matplotlib.pyplot as plt
import traceback
from scipy.ndimage import map_coordinates, gaussian_filter
import trimesh
import plotly.graph_objects as go
import time
import datetime
import pandas as pd
import json
from tqdm.notebook import tqdm
from typing import Dict, List, Tuple, Optional
import glob

try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not running in Colab or google.colab not found. Skipping Google Drive mount.")

def load_validation_pairs(
    sketch_dir: str,
    mesh_dir: str,
    limit: Optional[int] = None
) -> List[Dict]:
    import glob
    import os
    import re

    all_sketches = glob.glob(os.path.join(sketch_dir, "*.png"))

    model_sketches = {}
    for sketch_path in all_sketches:
        filename = os.path.basename(sketch_path)
        match = re.match(r'(dt_\d+)_(front|right|top)_sketch\.png', filename)

        if match:
            base_id = match.group(1)
            view_type = match.group(2)

            if base_id not in model_sketches:
                model_sketches[base_id] = {}

            model_sketches[base_id][view_type] = sketch_path

    valid_pairs = []
    for base_id, views in model_sketches.items():
        if all(view in views for view in ['front', 'right', 'top']):
            mesh_path = os.path.join(mesh_dir, f"{base_id}.obj")

            if os.path.exists(mesh_path):
                valid_pairs.append({
                    "base_name": base_id,
                    "front_sketch": views['front'],
                    "right_sketch": views['right'],
                    "top_sketch": views['top'],
                    "ground_truth_mesh": mesh_path
                })

    valid_pairs.sort(key=lambda x: x["base_name"])

    if limit is not None and len(valid_pairs) > limit:
        valid_pairs = valid_pairs[:limit]

    return valid_pairs

try:
    from transformers import AutoImageProcessor, AutoModel
    print("Transformers library found.")
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    print("Warning: transformers library not found. Using dummy classes.")
    print("Install using: pip install transformers")
    TRANSFORMERS_AVAILABLE = False

    class DummyAutoImageProcessor:
        @staticmethod
        def from_pretrained(name):
            print(f"Warning: Using dummy processor for {name}.")
            return lambda images, return_tensors: {'pixel_values': torch.randn(1, 3, 224, 224)}

    class DummyAutoModel:
        hidden_size = 768

        def __init__(self, name):
             print(f"Warning: Using dummy encoder for {name}.")
             self.dummy_param = nn.Parameter(torch.zeros(1))

        @staticmethod
        def from_pretrained(name):
            return DummyAutoModel(name)

        def __call__(self, pixel_values):
            batch_size = pixel_values.shape[0]
            seq_len = 16*16 + 1
            class DummyOutput:
                def __init__(self, last_hidden_state):
                    self.last_hidden_state = last_hidden_state
            return DummyOutput(last_hidden_state=torch.randn(batch_size, seq_len, self.hidden_size))

        def eval(self):
            pass

        def train(self):
             pass

        def parameters(self):
            yield self.dummy_param

        def named_parameters(self):
             yield ("dummy_param", self.dummy_param)

        @property
        def config(self):
            class DummyConfig:
                hidden_size = self.hidden_size
            return DummyConfig()

    if not TRANSFORMERS_AVAILABLE:
        AutoImageProcessor = DummyAutoImageProcessor
        AutoModel = DummyAutoModel

class PositionalEncoding2D(nn.Module):
    def __init__(self, d_model, height, width):
        super().__init__()
        if d_model % 4 != 0:
            raise ValueError(f"Cannot use sin/cos PE with odd dimension (got dim={d_model})")
        pe = torch.zeros(d_model, height, width)
        d_model_h = d_model // 2
        div_term_h = torch.exp(torch.arange(0., d_model_h, 2) * -(math.log(10000.0) / d_model_h))
        pos_h = torch.arange(0., height).unsqueeze(1)
        pe[0:d_model_h:2, :, :] = torch.sin(pos_h * div_term_h).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)
        pe[1:d_model_h:2, :, :] = torch.cos(pos_h * div_term_h).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)
        d_model_w = d_model // 2
        div_term_w = torch.exp(torch.arange(0., d_model_w, 2) * -(math.log(10000.0) / d_model_w))
        pos_w = torch.arange(0., width).unsqueeze(1)
        pe[d_model_h::2, :, :] = torch.sin(pos_w * div_term_w).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
        pe[d_model_h+1::2, :, :] = torch.cos(pos_w * div_term_w).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :, :x.size(2), :x.size(3)]
        return x

class TriplaneDecoder(nn.Module):
    def __init__(self, encoder_dim=768, decoder_dim=512, decoder_layers=6, decoder_heads=8,
                 output_channels=3, output_resolution=256, input_patch_grid_res=16):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.decoder_dim = decoder_dim
        self.output_resolution = output_resolution
        self.input_patch_grid_res = input_patch_grid_res
        self.num_patches = input_patch_grid_res * input_patch_grid_res

        self.input_proj = nn.Linear(encoder_dim, decoder_dim)
        self.pos_encoder = PositionalEncoding2D(decoder_dim, input_patch_grid_res, input_patch_grid_res)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=decoder_dim, nhead=decoder_heads, dim_feedforward=decoder_dim * 4,
            dropout=0.1, activation=F.relu, batch_first=True, norm_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=decoder_layers)

        num_upsample_stages = int(math.log2(output_resolution // input_patch_grid_res))
        if input_patch_grid_res * (2**num_upsample_stages) != output_resolution:
            raise ValueError("Output resolution must be a power-of-2 multiple of input_patch_grid_res")

        upsample_layers = []
        current_dim = decoder_dim
        for i in range(num_upsample_stages):
            out_dim = max(decoder_dim // (2**(i+1)), output_channels * 4, 16)
            upsample_layers.append(nn.Sequential(
                nn.ConvTranspose2d(current_dim, out_dim, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_dim), nn.ReLU(inplace=True),
                nn.Conv2d(out_dim, out_dim, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_dim), nn.ReLU(inplace=True)
            ))
            current_dim = out_dim
        self.upsample_neck = nn.Sequential(*upsample_layers)

        self.output_proj = nn.Conv2d(current_dim, output_channels, kernel_size=1, stride=1, padding=0)
        self.output_activation = nn.Identity()

    def forward(self, patch_embeddings_v1, patch_embeddings_v2, patch_embeddings_v3):
        projected_embeddings = []
        for pe in [patch_embeddings_v1, patch_embeddings_v2, patch_embeddings_v3]:
            if pe.shape[1] == self.num_patches + 1:
                pe_no_cls = pe[:, 1:, :]
            elif pe.shape[1] == self.num_patches:
                pe_no_cls = pe
            else:
                raise ValueError(f"Input seq len ({pe.shape[1]}) != patches ({self.num_patches}) or patches+1.")
            projected = self.input_proj(pe_no_cls)
            projected_embeddings.append(projected)

        combined_embeddings = projected_embeddings[0] + projected_embeddings[1] + projected_embeddings[2]
        batch_size = combined_embeddings.shape[0]

        spatial_input = combined_embeddings.permute(0, 2, 1).reshape(
            batch_size, self.decoder_dim, self.input_patch_grid_res, self.input_patch_grid_res)
        spatial_input_with_pos = self.pos_encoder(spatial_input)

        seq_with_pos = spatial_input_with_pos.flatten(2).permute(0, 2, 1)
        refined_seq = self.transformer_decoder(tgt=seq_with_pos, memory=seq_with_pos)

        spatial_features = refined_seq.permute(0, 2, 1).reshape(
            batch_size, self.decoder_dim, self.input_patch_grid_res, self.input_patch_grid_res)

        upsampled_features = self.upsample_neck(spatial_features)

        output_logits = self.output_proj(upsampled_features)
        output_triplane = self.output_activation(output_logits)

        return output_triplane

class ViTTriplaneGenerator(nn.Module):
    def __init__(self, encoder_model_name="facebook/dinov2-base", decoder_dim=512, decoder_layers=6,
                 decoder_heads=8, output_channels=3, output_resolution=256, input_patch_grid_res=16,
                 freeze_encoder=True):
        super().__init__()
        self.encoder_model_name = encoder_model_name
        self.output_resolution = output_resolution
        self.output_channels = output_channels

        print(f"Initializing encoder: {encoder_model_name}...")
        self.encoder = AutoModel.from_pretrained(encoder_model_name)

        encoder_output_dim = 768
        try:
            if hasattr(self.encoder, 'config') and hasattr(self.encoder.config, 'hidden_size'):
                 encoder_output_dim = self.encoder.config.hidden_size
            elif isinstance(self.encoder, DummyAutoModel):
                 if hasattr(DummyAutoModel, 'hidden_size'):
                      encoder_output_dim = DummyAutoModel.hidden_size
                      print(f"Using Dummy Encoder: Setting encoder_output_dim to {encoder_output_dim}.")
                 else:
                      print(f"Warning: DummyAutoModel class does not have hidden_size defined. Assuming {encoder_output_dim}.")
            else:
                 print(f"Warning: Could not automatically determine encoder hidden_size from config. Assuming {encoder_output_dim}.")
        except Exception as e:
             print(f"Warning: Error determining encoder dimension ({e}). Assuming {encoder_output_dim}.")

        print(f"Encoder output dimension assumed: {encoder_output_dim}")

        self.decoder = TriplaneDecoder(
            encoder_dim=encoder_output_dim, decoder_dim=decoder_dim, decoder_layers=decoder_layers,
            decoder_heads=decoder_heads, output_channels=output_channels, output_resolution=output_resolution,
            input_patch_grid_res=input_patch_grid_res)

        if freeze_encoder:
            self.freeze_encoder()
        else:
            self.unfreeze_encoder()

    def freeze_encoder(self):
        if hasattr(self, 'encoder') and isinstance(self.encoder, nn.Module):
            for param in self.encoder.parameters():
                param.requires_grad = False
            self.encoder.eval()

    def unfreeze_encoder(self):
         if hasattr(self, 'encoder') and isinstance(self.encoder, nn.Module):
            for param in self.encoder.parameters():
                param.requires_grad = True
            self.encoder.train()

    def forward(self, pixel_values_v1, pixel_values_v2, pixel_values_v3):
        if hasattr(self.encoder, 'eval'):
            self.encoder.eval()

        with torch.no_grad():
             outputs_v1 = self.encoder(pixel_values=pixel_values_v1)
             outputs_v2 = self.encoder(pixel_values=pixel_values_v2)
             outputs_v3 = self.encoder(pixel_values=pixel_values_v3)

        patch_embeddings_v1 = outputs_v1.last_hidden_state if hasattr(outputs_v1, 'last_hidden_state') else outputs_v1
        patch_embeddings_v2 = outputs_v2.last_hidden_state if hasattr(outputs_v2, 'last_hidden_state') else outputs_v2
        patch_embeddings_v3 = outputs_v3.last_hidden_state if hasattr(outputs_v3, 'last_hidden_state') else outputs_v3

        generated_triplane = self.decoder(patch_embeddings_v1, patch_embeddings_v2, patch_embeddings_v3)
        return generated_triplane

In [ ]:
def reconstruct_mesh_from_planes(plane_xy: np.ndarray,
                                 plane_yz: np.ndarray,
                                 plane_xz: np.ndarray,
                                 grid_resolution: int = 128,
                                 threshold: float = 0.5,
                                 combination_method: str = 'min',
                                 smoothing_sigma: float = 0.7,
                                 verbose: bool = True) -> trimesh.Trimesh | None:
    if verbose:
        print(f"\n--- Reconstructing model ---")
        print(f"Input plane shapes: XY:{plane_xy.shape}, YZ:{plane_yz.shape}, XZ:{plane_xz.shape}")
        print(f"Grid Resolution: {grid_resolution}, Threshold: {threshold}, Combination: {combination_method}")

    try:
        if not all(isinstance(p, np.ndarray) and p.ndim == 2 for p in [plane_xy, plane_yz, plane_xz]):
            print("Error: Input planes must be 2D NumPy arrays.")
            return None

        if not (plane_xy.shape == plane_yz.shape == plane_xz.shape):
             print(f"Warning: Plane shapes are different. Proceeding with XY shape as reference.")

        plane_res_h, plane_res_w = plane_xy.shape

        indices = np.arange(grid_resolution)
        x, y, z = np.meshgrid(indices, indices, indices, indexing='ij')
        coords_grid = np.stack([x, y, z], axis=-1).astype(float)

        coords_scaled = coords_grid.copy()
        if grid_resolution > 1:
            coords_scaled[..., 0] *= (plane_res_w - 1) / (grid_resolution - 1)
            coords_scaled[..., 1] *= (plane_res_h - 1) / (grid_resolution - 1)
            coords_scaled[..., 2] *= (plane_res_h - 1) / (grid_resolution - 1)
        else:
             coords_scaled[..., 0] = (plane_res_w - 1) / 2.0
             coords_scaled[..., 1] = (plane_res_h - 1) / 2.0
             coords_scaled[..., 2] = (plane_res_h - 1) / 2.0

        coords_scaled[..., 0] = np.clip(coords_scaled[..., 0], 0, plane_res_w - 1)
        coords_scaled[..., 1] = np.clip(coords_scaled[..., 1], 0, plane_res_h - 1)
        coords_scaled[..., 2] = np.clip(coords_scaled[..., 2], 0, plane_res_h - 1)

        coords_flat = coords_scaled.reshape(-1, 3).T

        if verbose:
            print("Sampling planes onto 3D grid...")

        normalized_coords = coords_scaled.copy()
        normalized_coords = normalized_coords / (plane_res_h - 1)

        x_norm = normalized_coords[..., 0].flatten()
        y_norm = normalized_coords[..., 1].flatten()
        z_norm = normalized_coords[..., 2].flatten()

        coords_xy = np.vstack([x_norm * (plane_res_w - 1), y_norm * (plane_res_h - 1)])
        coords_yz = np.vstack([y_norm * (plane_res_w - 1), z_norm * (plane_res_h - 1)])
        coords_xz = np.vstack([x_norm * (plane_res_w - 1), z_norm * (plane_res_h - 1)])

        sampled_xy = map_coordinates(plane_xy, coords_xy, order=1, mode='nearest').reshape(grid_resolution, grid_resolution, grid_resolution)
        sampled_yz = map_coordinates(plane_yz, coords_yz, order=1, mode='nearest').reshape(grid_resolution, grid_resolution, grid_resolution)
        sampled_xz = map_coordinates(plane_xz, coords_xz, order=1, mode='nearest').reshape(grid_resolution, grid_resolution, grid_resolution)

        if verbose:
            print(f"Combining sampled volumes using method: {combination_method}")

        if combination_method == 'average':
            volume = (sampled_xy + sampled_yz + sampled_xz) / 3.0
        elif combination_method == 'min':
            volume = np.minimum(np.minimum(sampled_xy, sampled_yz), sampled_xz)
        elif combination_method == 'max':
            volume = np.maximum(np.maximum(sampled_xy, sampled_yz), sampled_xz)
        elif combination_method == 'multiply':
            volume = sampled_xy * sampled_yz * sampled_xz
        else:
            print(f"Error: Unknown combination_method '{combination_method}'.")
            return None

        volume = gaussian_filter(volume, sigma=smoothing_sigma)

        if verbose:
            print(f"Volume grid created: shape={volume.shape}, Min={volume.min():.3f}, Max={volume.max():.3f}")

        pad_width = 1
        if grid_resolution > pad_width * 2:
            if verbose:
                print(f"Padding volume grid boundaries (width={pad_width})...")
            volume[:pad_width, :, :] = 0.0
            volume[-pad_width:, :, :] = 0.0
            volume[:, :pad_width, :] = 0.0
            volume[:, -pad_width:, :] = 0.0
            volume[:, :, :pad_width] = 0.0
            volume[:, :, -pad_width:] = 0.0

        if verbose:
            print(f"Running Marching Cubes (threshold={threshold})...")

        mesh = None
        try:
            mc_kwargs = {
                'matrix': volume,
                'pitch': 1.0,
                'threshold': threshold
            }
            mesh = trimesh.voxel.ops.matrix_to_marching_cubes(**mc_kwargs)

        except ValueError as ve:
            print(f"Marching Cubes ValueError: {ve}. Check threshold ({threshold}).")
            return None
        except Exception as e:
            print(f"Unexpected error during Marching Cubes: {e}")
            traceback.print_exc()
            return None

        if not isinstance(mesh, trimesh.Trimesh) or not mesh.vertices.size or not mesh.faces.size:
            print(f"MC resulted in empty or invalid mesh for threshold {threshold}.")
            return None

        scale_factor = 1.0 / grid_resolution
        mesh.vertices = (mesh.vertices * scale_factor) - 0.5

        try:
            mesh.process(validate=True)
        except Exception as proc_e:
            print(f"Warning: Error during mesh processing: {proc_e}")

        if verbose:
            print(f"Mesh generated and processed: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces.")

        return mesh

    except Exception as e:
        print(f"An unexpected error occurred during reconstruction: {e}")
        traceback.print_exc()
        return None

def align_to_sketch_axes(mesh: trimesh.Trimesh) -> trimesh.Trimesh:
    R_x = trimesh.transformations.rotation_matrix(
        np.radians(90), [1, 0, 0], point=None
    )
    mesh.apply_transform(R_x)
    R_z = trimesh.transformations.rotation_matrix(
        np.radians(-90), [0, 0, 1], point=None
    )
    mesh.apply_transform(R_z)
    return mesh

def sample_points_from_mesh(mesh: trimesh.Trimesh, n_samples: int = 10000) -> np.ndarray:
    samples, _ = trimesh.sample.sample_surface(mesh, n_samples)
    return samples

def chamfer_distance(points1: np.ndarray, points2: np.ndarray) -> Tuple[float, float]:
    points1 = np.asarray(points1)
    points2 = np.asarray(points2)

    distances1 = []
    for p1 in points1:
        distances = np.sqrt(np.sum((points2 - p1)**2, axis=1))
        distances1.append(np.min(distances))

    distances2 = []
    for p2 in points2:
        distances = np.sqrt(np.sum((points1 - p2)**2, axis=1))
        distances2.append(np.min(distances))

    chamfer_dist_mean = (np.mean(distances1) + np.mean(distances2)) / 2.0
    chamfer_dist_max = max(np.max(distances1), np.max(distances2))

    return chamfer_dist_mean, chamfer_dist_max

In [ ]:
# --- Configuration ---

# Device setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Paths
TRAIN_IMAGE_DIR_V1 = "/content/drive/MyDrive/Image_2D-3D_Source_folder/Transformer Dataset/sketch/front"
TRAIN_IMAGE_DIR_V2 = "/content/drive/MyDrive/Image_2D-3D_Source_folder/Transformer Dataset/sketch/right"
TRAIN_IMAGE_DIR_V3 = "/content/drive/MyDrive/Image_2D-3D_Source_folder/Transformer Dataset/sketch/top"
TRAIN_MESH_DIR = "/content/drive/MyDrive/Image_2D-3D_Source_folder/Dataset/3dmodel"

# Model checkpoint
CHECKPOINT_DIR = "/content/drive/MyDrive/Image_2D-3D_Source_folder/Colab/Transformer_training_checkpoint_sketch"
MODEL_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "latest_vit_voxel_triplane_decoder_0424_morning.pth")

# Output directory for evaluation results
OUTPUT_DIR = os.path.join(CHECKPOINT_DIR, "chamfer_evaluation")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model parameters
ENC_MODEL_NAME = "facebook/dinov2-base"
DEC_DIM = 512
DEC_LAYERS = 6
DEC_HEADS = 8
OUT_CH = 3
OUT_RES = 256
PATCH_GRID_RES = 16

# Reconstruction parameters
RECONSTRUCTION_GRID_RES = 128
RECONSTRUCTION_THRESHOLD = 0.5
RECONSTRUCTION_COMBO = 'min'

# Evaluation parameters
CHAMFER_N_SAMPLES = 10000

In [ ]:
def load_test_pairs(
    sketch_front_dir: str,
    sketch_right_dir: str,
    sketch_top_dir: str,
    mesh_dir: str,
    limit: Optional[int] = 100
) -> List[Dict]:
    front_sketches = glob.glob(os.path.join(sketch_front_dir, "*.png"))

    test_pairs = []
    for front_path in front_sketches:
        base_name = os.path.splitext(os.path.basename(front_path))[0]

        right_path = os.path.join(sketch_right_dir, f"{base_name}.png")
        top_path = os.path.join(sketch_top_dir, f"{base_name}.png")
        mesh_path = os.path.join(mesh_dir, f"{base_name}.obj")

        if os.path.exists(right_path) and os.path.exists(top_path) and os.path.exists(mesh_path):
            test_pairs.append({
                "front_sketch": front_path,
                "right_sketch": right_path,
                "top_sketch": top_path,
                "ground_truth_mesh": mesh_path,
                "base_name": base_name
            })

    if limit is not None and len(test_pairs) > limit:
        test_pairs = test_pairs[:limit]

    return test_pairs

def run_chamfer_evaluation(
    sketch_front_dir=TRAIN_IMAGE_DIR_V1,
    sketch_right_dir=TRAIN_IMAGE_DIR_V2,
    sketch_top_dir=TRAIN_IMAGE_DIR_V3,
    mesh_dir=TRAIN_MESH_DIR,
    limit=None,
    n_samples=CHAMFER_N_SAMPLES,
    reconstruction_grid_res=RECONSTRUCTION_GRID_RES,
    reconstruction_threshold=RECONSTRUCTION_THRESHOLD,
    reconstruction_combo=RECONSTRUCTION_COMBO
):
    print("=" * 70)
    print("Starting Chamfer Distance Evaluation")
    print("=" * 70)

    print("\nLoading model...")
    try:
        image_processor = AutoImageProcessor.from_pretrained(ENC_MODEL_NAME)

        model = ViTTriplaneGenerator(
            encoder_model_name=ENC_MODEL_NAME,
            decoder_dim=DEC_DIM,
            decoder_layers=DEC_LAYERS,
            decoder_heads=DEC_HEADS,
            output_channels=OUT_CH,
            output_resolution=OUT_RES,
            input_patch_grid_res=PATCH_GRID_RES,
            freeze_encoder=True
        ).to(DEVICE)

        if os.path.exists(MODEL_CHECKPOINT_PATH):
            model.decoder.load_state_dict(torch.load(MODEL_CHECKPOINT_PATH, map_location=DEVICE))
            print(f"Loaded decoder weights from: {MODEL_CHECKPOINT_PATH}")
        else:
            print(f"ERROR: Decoder weights not found at {MODEL_CHECKPOINT_PATH}")
            return None

        model.eval()
        print("Model loaded and set to evaluation mode.")

    except Exception as e:
        print(f"Error loading model: {e}")
        traceback.print_exc()
        return None

    print(f"\nLoading test pairs from: {sketch_front_dir}, {sketch_right_dir}, {sketch_top_dir}, {mesh_dir}")
    test_pairs = load_test_pairs(
        sketch_front_dir=sketch_front_dir,
        sketch_right_dir=sketch_right_dir,
        sketch_top_dir=sketch_top_dir,
        mesh_dir=mesh_dir,
        limit=limit
    )

    print(f"Found {len(test_pairs)} valid test pairs")

    if len(test_pairs) == 0:
        print("No valid test pairs found. Exiting.")
        return None

    results = []

    for pair_idx, pair in enumerate(tqdm(test_pairs, desc="Evaluating Test Pairs")):
        base_name = pair["base_name"]
        try:
            print(f"\nProcessing {base_name} ({pair_idx+1}/{len(test_pairs)})...")

            gt_mesh = trimesh.load(pair["ground_truth_mesh"], force='mesh')

            img_front = Image.open(pair["front_sketch"]).convert("RGB")
            img_right = Image.open(pair["right_sketch"]).convert("RGB")
            img_top = Image.open(pair["top_sketch"]).convert("RGB")

            processed_front = image_processor(images=img_front, return_tensors="pt")['pixel_values'].to(DEVICE)
            processed_right = image_processor(images=img_right, return_tensors="pt")['pixel_values'].to(DEVICE)
            processed_top = image_processor(images=img_top, return_tensors="pt")['pixel_values'].to(DEVICE)

            start_time = time.time()
            with torch.no_grad():
                predicted_triplane_logits = model(processed_front, processed_right, processed_top)

            predicted_triplane_sigmoid = torch.sigmoid(predicted_triplane_logits)

            triplane_np = predicted_triplane_sigmoid[0].cpu().numpy()

            plane_xy_pred = triplane_np[0]
            plane_yz_pred = triplane_np[1]
            plane_xz_pred = triplane_np[2]

            reconstructed_mesh = reconstruct_mesh_from_planes(
                plane_xy=plane_xy_pred,
                plane_yz=plane_yz_pred,
                plane_xz=plane_xz_pred,
                grid_resolution=reconstruction_grid_res,
                threshold=reconstruction_threshold,
                combination_method=reconstruction_combo,
                verbose=False
            )

            inference_time = time.time() - start_time

            if reconstructed_mesh is None or len(reconstructed_mesh.vertices) == 0 or len(reconstructed_mesh.faces) == 0:
                print(f"Failed to reconstruct valid mesh for {base_name}. Skipping.")
                continue

            reconstructed_mesh = align_to_sketch_axes(reconstructed_mesh.copy())

            gt_points = sample_points_from_mesh(gt_mesh, n_samples)
            pred_points = sample_points_from_mesh(reconstructed_mesh, n_samples)

            cd_mean, cd_max = chamfer_distance(gt_points, pred_points)

            result = {
                "base_name": base_name,
                "chamfer_dist_mean": cd_mean,
                "chamfer_dist_max": cd_max,
                "inference_time": inference_time,
                "gt_vertices": len(gt_mesh.vertices),
                "gt_faces": len(gt_mesh.faces),
                "pred_vertices": len(reconstructed_mesh.vertices),
                "pred_faces": len(reconstructed_mesh.faces)
            }
            results.append(result)

            print(f"Results for {base_name}:")
            print(f"  Mean Chamfer Distance: {cd_mean:.6f}")
            print(f"  Max Chamfer Distance: {cd_max:.6f}")
            print(f"  Inference Time: {inference_time:.2f}s")
            print(f"  GT Mesh: {len(gt_mesh.vertices)} vertices, {len(gt_mesh.faces)} faces")
            print(f"  Pred Mesh: {len(reconstructed_mesh.vertices)} vertices, {len(reconstructed_mesh.faces)} faces")

            save_reconstructed = True
            if save_reconstructed:
                reconstruction_dir = os.path.join(OUTPUT_DIR, "reconstructions")
                os.makedirs(reconstruction_dir, exist_ok=True)
                mesh_save_path = os.path.join(reconstruction_dir, f"{base_name}_reconstructed.obj")
                reconstructed_mesh.export(mesh_save_path)

        except Exception as e:
            print(f"Error evaluating {base_name}: {e}")
            traceback.print_exc()

    if not results:
        print("No successful evaluations. Cannot create results.")
        return None

    results_df = pd.DataFrame(results)

    summary = {
        "mean_chamfer_dist": results_df["chamfer_dist_mean"].mean(),
        "median_chamfer_dist": results_df["chamfer_dist_mean"].median(),
        "min_chamfer_dist": results_df["chamfer_dist_mean"].min(),
        "max_chamfer_dist": results_df["chamfer_dist_mean"].max(),
        "std_chamfer_dist": results_df["chamfer_dist_mean"].std(),
        "mean_inference_time": results_df["inference_time"].mean(),
        "total_models_evaluated": len(results_df),
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    print("\n===== Evaluation Summary =====")
    for key, value in summary.items():
        print(f"{key}: {value}")

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    results_csv_path = os.path.join(OUTPUT_DIR, f"chamfer_eval_{timestamp}.csv")
    results_df.to_csv(results_csv_path, index=False)

    summary_json_path = os.path.join(OUTPUT_DIR, f"chamfer_eval_summary_{timestamp}.json")
    with open(summary_json_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"\nDetailed results saved to: {results_csv_path}")
    print(f"Summary saved to: {summary_json_path}")

    try:
        plt.figure(figsize=(10, 6))
        plt.hist(results_df["chamfer_dist_mean"], bins=20, alpha=0.7)
        plt.axvline(summary["mean_chamfer_dist"], color='r', linestyle='--', label=f'Mean: {summary["mean_chamfer_dist"]:.4f}')
        plt.axvline(summary["median_chamfer_dist"], color='g', linestyle='--', label=f'Median: {summary["median_chamfer_dist"]:.4f}')
        plt.xlabel('Mean Chamfer Distance')
        plt.ylabel('Count')
        plt.title('Distribution of Chamfer Distances')
        plt.legend()
        plt.tight_layout()

        plot_path = os.path.join(OUTPUT_DIR, f"chamfer_dist_histogram_{timestamp}.png")
        plt.savefig(plot_path)
        plt.close()

        print(f"Distribution plot saved to: {plot_path}")
    except Exception as e:
        print(f"Error generating visualization: {e}")

    return results_df, summary

def run_validation_evaluation(
    validation_sketch_dir="/content/drive/MyDrive/Image_2D-3D_Source_folder/Val_Dataset/sketch",
    validation_mesh_dir="/content/drive/MyDrive/Image_2D-3D_Source_folder/Val_Dataset/3dmodel",
    limit=None,
    n_samples=CHAMFER_N_SAMPLES,
    reconstruction_grid_res=RECONSTRUCTION_GRID_RES,
    reconstruction_threshold=RECONSTRUCTION_THRESHOLD,
    reconstruction_combo=RECONSTRUCTION_COMBO
):
    print("=" * 70)
    print("Starting Validation Set Chamfer Distance Evaluation")
    print("=" * 70)

    print("\nLoading model...")
    try:
        image_processor = AutoImageProcessor.from_pretrained(ENC_MODEL_NAME)

        model = ViTTriplaneGenerator(
            encoder_model_name=ENC_MODEL_NAME,
            decoder_dim=DEC_DIM,
            decoder_layers=DEC_LAYERS,
            decoder_heads=DEC_HEADS,
            output_channels=OUT_CH,
            output_resolution=OUT_RES,
            input_patch_grid_res=PATCH_GRID_RES,
            freeze_encoder=True
        ).to(DEVICE)

        if os.path.exists(MODEL_CHECKPOINT_PATH):
            model.decoder.load_state_dict(torch.load(MODEL_CHECKPOINT_PATH, map_location=DEVICE))
            print(f"Loaded decoder weights from: {MODEL_CHECKPOINT_PATH}")
        else:
            print(f"ERROR: Decoder weights not found at {MODEL_CHECKPOINT_PATH}")
            return None

        model.eval()
        print("Model loaded and set to evaluation mode.")

    except Exception as e:
        print(f"Error loading model: {e}")
        traceback.print_exc()
        return None

    print(f"\nLoading validation test pairs from: {validation_sketch_dir}, {validation_mesh_dir}")
    test_pairs = load_validation_pairs(
        sketch_dir=validation_sketch_dir,
        mesh_dir=validation_mesh_dir,
        limit=limit
    )

    print(f"Found {len(test_pairs)} valid validation test pairs")

    if len(test_pairs) == 0:
        print("No valid validation test pairs found. Exiting.")
        return None

    results = []

    for pair_idx, pair in enumerate(tqdm(test_pairs, desc="Evaluating Validation Pairs")):
        base_name = pair["base_name"]
        try:
            print(f"\nProcessing {base_name} ({pair_idx+1}/{len(test_pairs)})...")

            gt_mesh = trimesh.load(pair["ground_truth_mesh"], force='mesh')

            img_front = Image.open(pair["front_sketch"]).convert("RGB")
            img_right = Image.open(pair["right_sketch"]).convert("RGB")
            img_top = Image.open(pair["top_sketch"]).convert("RGB")

            processed_front = image_processor(images=img_front, return_tensors="pt")['pixel_values'].to(DEVICE)
            processed_right = image_processor(images=img_right, return_tensors="pt")['pixel_values'].to(DEVICE)
            processed_top = image_processor(images=img_top, return_tensors="pt")['pixel_values'].to(DEVICE)

            start_time = time.time()
            with torch.no_grad():
                predicted_triplane_logits = model(processed_front, processed_right, processed_top)

            predicted_triplane_sigmoid = torch.sigmoid(predicted_triplane_logits)

            triplane_np = predicted_triplane_sigmoid[0].cpu().numpy()

            plane_xy_pred = triplane_np[0]
            plane_yz_pred = triplane_np[1]
            plane_xz_pred = triplane_np[2]

            reconstructed_mesh = reconstruct_mesh_from_planes(
                plane_xy=plane_xy_pred,
                plane_yz=plane_yz_pred,
                plane_xz=plane_xz_pred,
                grid_resolution=reconstruction_grid_res,
                threshold=reconstruction_threshold,
                combination_method=reconstruction_combo,
                verbose=False
            )

            inference_time = time.time() - start_time

            if reconstructed_mesh is None or len(reconstructed_mesh.vertices) == 0 or len(reconstructed_mesh.faces) == 0:
                print(f"Failed to reconstruct valid mesh for {base_name}. Skipping.")
                continue

            reconstructed_mesh = align_to_sketch_axes(reconstructed_mesh.copy())

            gt_points = sample_points_from_mesh(gt_mesh, n_samples)
            pred_points = sample_points_from_mesh(reconstructed_mesh, n_samples)

            cd_mean, cd_max = chamfer_distance(gt_points, pred_points)

            result = {
                "base_name": base_name,
                "chamfer_dist_mean": cd_mean,
                "chamfer_dist_max": cd_max,
                "inference_time": inference_time,
                "gt_vertices": len(gt_mesh.vertices),
                "gt_faces": len(gt_mesh.faces),
                "pred_vertices": len(reconstructed_mesh.vertices),
                "pred_faces": len(reconstructed_mesh.faces)
            }
            results.append(result)

            print(f"Results for {base_name}:")
            print(f"  Mean Chamfer Distance: {cd_mean:.6f}")
            print(f"  Max Chamfer Distance: {cd_max:.6f}")
            print(f"  Inference Time: {inference_time:.2f}s")
            print(f"  GT Mesh: {len(gt_mesh.vertices)} vertices, {len(gt_mesh.faces)} faces")
            print(f"  Pred Mesh: {len(reconstructed_mesh.vertices)} vertices, {len(reconstructed_mesh.faces)} faces")

            save_reconstructed = True
            if save_reconstructed:
                reconstruction_dir = os.path.join(OUTPUT_DIR, "validation_reconstructions")
                os.makedirs(reconstruction_dir, exist_ok=True)
                mesh_save_path = os.path.join(reconstruction_dir, f"{base_name}_reconstructed.obj")
                reconstructed_mesh.export(mesh_save_path)

        except Exception as e:
            print(f"Error evaluating {base_name}: {e}")
            traceback.print_exc()

    if not results:
        print("No successful evaluations. Cannot create results.")
        return None

    results_df = pd.DataFrame(results)

    summary = {
        "dataset": "validation",
        "mean_chamfer_dist": results_df["chamfer_dist_mean"].mean(),
        "median_chamfer_dist": results_df["chamfer_dist_mean"].median(),
        "min_chamfer_dist": results_df["chamfer_dist_mean"].min(),
        "max_chamfer_dist": results_df["chamfer_dist_mean"].max(),
        "std_chamfer_dist": results_df["chamfer_dist_mean"].std(),
        "mean_inference_time": results_df["inference_time"].mean(),
        "total_models_evaluated": len(results_df),
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    print("\n===== Validation Evaluation Summary =====")
    for key, value in summary.items():
        print(f"{key}: {value}")

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    results_csv_path = os.path.join(OUTPUT_DIR, f"validation_chamfer_eval_{timestamp}.csv")
    results_df.to_csv(results_csv_path, index=False)

    summary_json_path = os.path.join(OUTPUT_DIR, f"validation_chamfer_eval_summary_{timestamp}.json")
    with open(summary_json_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"\nDetailed validation results saved to: {results_csv_path}")
    print(f"Validation summary saved to: {summary_json_path}")

    return results_df, summary

results_df, summary = run_validation_evaluation(
    validation_sketch_dir="/content/drive/MyDrive/Image_2D-3D_Source_folder/Val_Dataset/sketch",
    validation_mesh_dir="/content/drive/MyDrive/Image_2D-3D_Source_folder/Val_Dataset/3dmodel",
    limit=None
)

print("\nTop 5 Best Examples (Lowest Chamfer Distance):")
print(results_df.sort_values("chamfer_dist_mean").head(5)[["base_name", "chamfer_dist_mean"]])

print("\nTop 5 Worst Examples (Highest Chamfer Distance):")
print(results_df.sort_values("chamfer_dist_mean", ascending=False).head(5)[["base_name", "chamfer_dist_mean"]])

plt.figure(figsize=(10, 6))
plt.scatter(results_df["gt_vertices"], results_df["chamfer_dist_mean"], alpha=0.7)
plt.xlabel('Number of GT Vertices')
plt.ylabel('Mean Chamfer Distance')
plt.title('Relationship Between Mesh Complexity and Reconstruction Quality')
plt.tight_layout()
plt.show()

def visualize_comparison(base_name, show_gt=True, show_pred=True):
    validation_mesh_dir = "/content/drive/MyDrive/Image_2D-3D_Source_folder/Val_Dataset/3dmodel"
    validation_sketch_dir = "/content/drive/MyDrive/Image_2D-3D_Source_folder/Val_Dataset/sketch"

    gt_path = os.path.join(validation_mesh_dir, f"{base_name}.obj")
    pred_path = os.path.join(OUTPUT_DIR, "validation_reconstructions", f"{base_name}_reconstructed.obj")

    front_path = os.path.join(validation_sketch_dir, f"{base_name}_front_sketch.png")
    right_path = os.path.join(validation_sketch_dir, f"{base_name}_right_sketch.png")
    top_path = os.path.join(validation_sketch_dir, f"{base_name}_top_sketch.png")

    gt_mesh = None
    pred_mesh = None

    if show_gt:
        try:
            gt_mesh = trimesh.load(gt_path, force='mesh')
        except Exception as e:
            print(f"Error loading GT mesh: {e}")
            show_gt = False

    if show_pred:
        try:
            pred_mesh = trimesh.load(pred_path, force='mesh')
        except Exception as e:
            print(f"Error loading predicted mesh: {e}")
            show_pred = False

    num_cols = 2 if (show_gt and show_pred) else 1
    fig = plt.figure(figsize=(12, 10))

    sketch_row = fig.add_gridspec(1, 3, left=0.1, right=0.9, top=0.9, bottom=0.7)
    for i, path in enumerate([front_path, right_path, top_path]):
        ax = fig.add_subplot(sketch_row[0, i])
        if os.path.exists(path):
            ax.imshow(plt.imread(path))
            ax.set_title(['Front View', 'Right View', 'Top View'][i])
        else:
            ax.text(0.5, 0.5, 'Sketch not found', ha='center', va='center')
        ax.axis('off')

    mesh_row = fig.add_gridspec(1, num_cols, left=0.1, right=0.9, top=0.65, bottom=0.1)

    if show_gt:
        ax_gt = fig.add_subplot(mesh_row[0, 0 if num_cols == 1 else 0], projection='3d')
        ax_gt.set_title('Ground Truth Mesh')

        verts = gt_mesh.vertices
        faces = gt_mesh.faces
        for i in range(len(faces)):
            face = faces[i]
            for j in range(3):
                ax_gt.plot3D([verts[face[j], 0], verts[face[(j+1)%3], 0]],
                             [verts[face[j], 1], verts[face[(j+1)%3], 1]],
                             [verts[face[j], 2], verts[face[(j+1)%3], 2]],
                             'k-', alpha=0.2)

        ax_gt.set_xlim([-0.6, 0.6])
        ax_gt.set_ylim([-0.6, 0.6])
        ax_gt.set_zlim([-0.6, 0.6])
        ax_gt.set_box_aspect([1, 1, 1])

    if show_pred:
        ax_pred = fig.add_subplot(mesh_row[0, 0 if num_cols == 1 else 1], projection='3d')
        ax_pred.set_title('Predicted Mesh')

        verts = pred_mesh.vertices
        faces = pred_mesh.faces
        for i in range(len(faces)):
            face = faces[i]
            for j in range(3):
                ax_pred.plot3D([verts[face[j], 0], verts[face[(j+1)%3], 0]],
                               [verts[face[j], 1], verts[face[(j+1)%3], 1]],
                               [verts[face[j], 2], verts[face[(j+1)%3], 2]],
                               'r-', alpha=0.2)

        ax_pred.set_xlim([-0.6, 0.6])
        ax_pred.set_ylim([-0.6, 0.6])
        ax_pred.set_zlim([-0.6, 0.6])
        ax_pred.set_box_aspect([1, 1, 1])

    plt.suptitle(f'Example: {base_name}', fontsize=16)
    plt.tight_layout()
    plt.show()

if len(results_df) > 0:
    best_example = results_df.sort_values("chamfer_dist_mean").iloc[0]["base_name"]
    print(f"Visualizing best example: {best_example}")
    visualize_comparison(best_example)

    worst_example = results_df.sort_values("chamfer_dist_mean", ascending=False).iloc[0]["base_name"]
    print(f"Visualizing worst example: {worst_example}")
    visualize_comparison(worst_example)